# 🚀 Simulación de Gemelo Digital e IIoT para Monitoreo Predictivo de Maquinaria Industrial
**Asignatura:** Teoría de Sistemas (TSI115) - Universidad de El Salvador  
**Proyecto:** Tarea de Investigación - Etapa 2  
**Tema:** Diseño de un Gemelo Digital basado en IoT para el monitoreo predictivo del rendimiento en maquinaria industrial  
---
### 📌 Descripción del Cuaderno
Este cuaderno de Jupyter / Google Colab implementa la simulación computacional del **Gemelo Digital** para un activo rotativo (motor/rodamiento industrial), integrando los principios de la **Teoría General de Sistemas (TGS)** y el estándar **ISO 23247**:
1. **Entidad Observable (OME):** Emulación de telemetría IoT (vibración triaxial y temperatura) con degradación sintética progresiva (falla de rodamiento BPFO).
2. **Entidad de Gemelo Digital (DTE - Caja Blanca):** Procesamiento de señales en el dominio del tiempo (RMS, Kurtosis) y frecuencia mediante **Transformada Rápida de Fourier (FFT)**.
3. **Evaluación de Severidad (ISO 10816):** Clasificación del estado operativo por zonas de alarma.
4. **Cibernética y Retroalimentación (Feedback Loop):** Estimación de Vida Útil Remanente (RUL), Índice de Salud ($HI$) y disparo de alertas automáticas para mantener la **homeostasis** del sistema.


In [ ]:
# ========================================== #
# 1. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN #
# ========================================== #
import numpy as np
import pandas as pd
import scipy.fft as fft
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Estilo visual para los gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
print('✅ Librerías cargadas exitosamente.')


## 📡 1. Emulador de Telemetría IoT (Entidad OME)
Generamos un dataset simulado que representa la evolución temporal de la telemetría de un rodamiento industrial durante **100 días** de operación continua (simulando datos estilo *NASA IMS Bearing Dataset*).
* **Frecuencia de muestreo ($F_s$):** 20,000 Hz
* **Velocidad de rotación ($1X$):** 1770 RPM ($29.5$ Hz)
* **Frecuencia característica de falla BPFO (*Outer Race*):** $236.2$ Hz


In [ ]:
# ========================================== #
# 2. GENERACIÓN DE TELEMETRÍA SINTÉTICA IoT   #
# ========================================== #
np.random.seed(42)

# Parámetros físicos del sistema
fs = 20000          # Frecuencia de muestreo (Hz)
duracion_muestra = 0.5 # Segundos por captura de telemetría
num_muestras_t = int(fs * duracion_muestra)
t = np.linspace(0, duracion_muestra, num_muestras_t, endpoint=False)

# Frecuencias mecánicas características
f_rot = 1770 / 60.0  # Frecuencia de rotación (1X RPM = 29.5 Hz)
f_bpfo = 236.2       # Frecuencia de falla en pista externa (BPFO = 236.2 Hz)

# Simulación a lo largo de 100 días de operación
dias = np.arange(1, 101)
historial_rms = []
historial_temp = []
historial_kurtosis = []
historial_espectros = []
muestras_tiempo_seleccionadas = {}

for d in dias:
    # Factor de degradación exponencial (se acelera a partir del día 40)
    if d < 40:
        degradacion = 0.05 + 0.08 * (d / 40)
    elif d < 70:
        degradacion = 0.13 + 0.65 * ((d - 40) / 30)**1.5
    else:
        degradacion = 0.78 + 2.50 * ((d - 70) / 30)**2.0
    
    # Componentes de la señal física:
    # 1. Señal normal de rotación 1X y armónico 2X
    s_normal = 0.6 * np.sin(2 * np.pi * f_rot * t) + 0.25 * np.sin(2 * np.pi * 2 * f_rot * t)
    
    # 2. Impactos de la falla en desarrollo (BPFO + Modulación por rotación)
    s_falla = degradacion * np.sin(2 * np.pi * f_bpfo * t) * (1 + 0.5 * np.sin(2 * np.pi * f_rot * t))
    
    # 3. Ruido gaussiano blanco del entorno industrial
    ruido = np.random.normal(0, 0.15, num_muestras_t)
    
    # Señal total de aceleración (g)
    senos_acc = s_normal + s_falla + ruido
    
    # Conversión escalada a velocidad RMS (mm/s) para evaluación ISO 10816
    acc_rms_g = np.sqrt(np.mean(senos_acc**2))
    v_rms_mms = (acc_rms_g * 9.81 * 1000 / (2 * np.pi * f_rot)) * 0.15
    
    # Medición de temperatura (°C) correlacionada a la fricción
    temp_base = 42.0 + 0.15 * d + 18.0 * (degradacion**1.2) + np.random.normal(0, 0.4)
    
    # Kurtosis (indicador estadístico de impulsividad)
    kurt = np.mean((senos_acc - np.mean(senos_acc))**4) / (np.std(senos_acc)**4)
    
    # Guardar métricas
    historial_rms.append(v_rms_mms)
    historial_temp.append(temp_base)
    historial_kurtosis.append(kurt)
    
    # Guardar muestras para el análisis FFT de días clave
    if d in [5, 50, 75, 95]:
        muestras_tiempo_seleccionadas[d] = (t, senos_acc)

print(f'✅ Telemetría emulada correctamente para {len(dias)} días de operación.')


## 🔬 2. Procesamiento en la Entidad de Gemelo Digital (DTE - Caja Blanca)
Aplicamos la **Transformada Rápida de Fourier (FFT)** a las señales de vibración para descomponer la señal temporal en el espectro de frecuencias $G(f)$. Esto permite pasar de un enfoque de **caja negra** a una **caja blanca**, identificando la firma espectral de la falla BPFO en $236.2$ Hz.


In [ ]:
# ========================================== #
# 3. ANALÍTICA DE SEÑAL Y CÁLCULO DE FFT     #
# ========================================== #
def calcular_fft(senos_acc, fs):
    N = len(senos_acc)
    # FFT unilateral
    yf = fft.fft(senos_acc)
    xf = fft.fftfreq(N, 1 / fs)[:N//2]
    amplitud = 2.0 / N * np.abs(yf[0:N//2])
    return xf, amplitud

print('✅ Función de cálculo FFT definida.')


## 🛡️ 3. Evaluador de Severidad ISO 10816, RUL y Lazo de Retroalimentación
Se evalúa la salud del activo utilizando los umbrales de la norma **ISO 10816-3** para maquinaria industrial de tamaño mediano (Clase II):
* **Zona A (Bueno):** $< 1.8$ mm/s RMS (Verde)
* **Zona B (Aceptable):** $1.8 - 4.5$ mm/s RMS (Azul)
* **Zona C (Alarma / Advertencia):** $4.5 - 7.1$ mm/s RMS (Amarillo/Naranja)
* **Zona D (Peligro / Inaceptable):** $> 7.1$ mm/s RMS (Rojo)

Además, se calcula el **Índice de Salud ($HI$)** y la **Vida Útil Remanente ($RUL$)** para activar la **retroalimentación negativa cibernética**.


In [ ]:
# ========================================== #
# 4. EVALUACIÓN ISO 10816 Y CÁLCULO DE RUL    #
# ========================================== #
df_telemetria = pd.DataFrame({
    'Dia': dias,
    'Vibracion_RMS_mms': historial_rms,
    'Temperatura_C': historial_temp,
    'Kurtosis': historial_kurtosis
})

# Clasificación según ISO 10816-3
def clasificar_iso(rms):
    if rms < 1.8:
        return 'Zona A (Bueno)'
    elif rms < 4.5:
        return 'Zona B (Aceptable)'
    elif rms < 7.1:
        return 'Zona C (Alarma)'
    else:
        return 'Zona D (Peligro)'

df_telemetria['Estado_ISO'] = df_telemetria['Vibracion_RMS_mms'].apply(clasificar_iso)

# Índice de Salud HI (100% = Sano, 0% = Falla Crítica a 9.0 mm/s)
rms_limite_critico = 9.0
df_telemetria['Indice_Salud_HI'] = np.maximum(0, np.minimum(100, 100 * (1 - (df_telemetria['Vibracion_RMS_mms'] - 1.0) / (rms_limite_critico - 1.0))))

# Estimación de Vida Útil Remanente (RUL en Días)
df_telemetria['RUL_Estimado_Dias'] = np.maximum(0, np.round((100 - df_telemetria['Dia']) * (df_telemetria['Indice_Salud_HI'] / 100)))

# Identificar el día exacto de disparo del Lazo de Retroalimentación (Primera Alerta Zona C)
dia_alerta_row = df_telemetria[df_telemetria['Vibracion_RMS_mms'] >= 4.5]
dia_alerta = int(dia_alerta_row['Dia'].iloc[0]) if len(dia_alerta_row) > 0 else None

dia_critico_row = df_telemetria[df_telemetria['Vibracion_RMS_mms'] >= 7.1]
dia_critico = int(dia_critico_row['Dia'].iloc[0]) if len(dia_critico_row) > 0 else None

print(f'🚨 Disparo de Alerta Cibernética (Zona C - Retroalimentación): Día {dia_alerta}')
print(f'🛑 Estado Crítico de Falla (Zona D - Parada Recomendada): Día {dia_critico}')


## 📊 4. Dashboard de Visualización del Gemelo Digital (Entidad UE)
A continuación se genera la interfaz del **Gemelo Digital**, mostrando la evolución temporal de los indicadores y el diagnóstico espectral por FFT.


In [ ]:
# ========================================== #
# 5. GENERACIÓN DEL DASHBOARD MULTI-PANEL    #
# ========================================== #
fig = plt.figure(figsize=(16, 12))
plt.suptitle('DASHBOARD DE MONITOREO PREDICTIVO - GEMELO DIGITAL IIoT (ISO 23247 / TGS)', fontsize=14, fontweight='bold', y=0.98)

# Panel 1: Señales en el Dominio del Tiempo (Día 5 vs Día 95)
ax1 = fig.add_subplot(2, 2, 1)
t_sample, acc_d5 = muestras_tiempo_seleccionadas[5]
_, acc_d95 = muestras_tiempo_seleccionadas[95]
ax1.plot(t_sample[:1000]*1000, acc_d5[:1000], label='Día 5 (Estado Sano)', color='mediumseagreen', alpha=0.85)
ax1.plot(t_sample[:1000]*1000, acc_d95[:1000], label='Día 95 (Falla Avanzada)', color='firebrick', alpha=0.85)
ax1.set_title('1. Señal Temporal de Vibración (Aceleración g)', fontweight='bold')
ax1.set_xlabel('Tiempo (ms)')
ax1.set_ylabel('Aceleración (g)')
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.6)

# Panel 2: Espectro de Frecuencias FFT (Aislamiento de Falla BPFO)
ax2 = fig.add_subplot(2, 2, 2)
xf, yf_d5 = calcular_fft(acc_d5, fs)
_, yf_d95 = calcular_fft(acc_d95, fs)
ax2.plot(xf, yf_d5, label='Día 5 (Sano)', color='mediumseagreen', alpha=0.7)
ax2.plot(xf, yf_d95, label='Día 95 (Degradado)', color='crimson', alpha=0.85)
ax2.axvline(x=f_bpfo, color='purple', linestyle='--', linewidth=1.5, label=f'Falla BPFO ({f_bpfo} Hz)')
ax2.axvline(x=f_rot, color='blue', linestyle=':', linewidth=1.2, label=f'1X Rotación ({f_rot:.1f} Hz)')
ax2.set_xlim(0, 500)  # Enfocar frecuencias de interés (0 - 500 Hz)
ax2.set_title('2. Espectro FFT - Diagnóstico de Caja Blanca', fontweight='bold')
ax2.set_xlabel('Frecuencia (Hz)')
ax2.set_ylabel('Amplitud (g)')
ax2.legend(loc='upper right')
ax2.grid(True, linestyle='--', alpha=0.6)

# Panel 3: Tendencia RMS y Umbrales ISO 10816
ax3 = fig.add_subplot(2, 2, 3)
ax3.plot(df_telemetria['Dia'], df_telemetria['Vibracion_RMS_mms'], color='royalblue', linewidth=2, label='Vibración RMS (mm/s)')
ax3.axhline(y=1.8, color='green', linestyle='--', alpha=0.7, label='Límite Zona A/B (1.8 mm/s)')
ax3.axhline(y=4.5, color='orange', linestyle='--', alpha=0.8, label='Umbral Alerta ISO (4.5 mm/s)')
ax3.axhline(y=7.1, color='red', linestyle='--', alpha=0.8, label='Umbral Peligro ISO (7.1 mm/s)')
if dia_alerta:
    ax3.axvline(x=dia_alerta, color='darkorange', linestyle=':', linewidth=2, label=f'Alerta Retroalimentación (Día {dia_alerta})')
ax3.set_title('3. Evolución de Vibración RMS y Umbrales ISO 10816', fontweight='bold')
ax3.set_xlabel('Tiempo de Operación (Días)')
ax3.set_ylabel('Velocidad RMS (mm/s)')
ax3.legend(loc='upper left', fontsize=8)
ax3.grid(True, linestyle='--', alpha=0.6)

# Panel 4: Índice de Salud HI (%) y RUL
ax4 = fig.add_subplot(2, 2, 4)
ax4_temp = ax4.twinx()
p1 = ax4.plot(df_telemetria['Dia'], df_telemetria['Indice_Salud_HI'], color='teal', linewidth=2, label='Índice de Salud HI (%)')
p2 = ax4_temp.plot(df_telemetria['Dia'], df_telemetria['Temperatura_C'], color='coral', linewidth=1.8, linestyle='-.', label='Temperatura (°C)')
ax4.set_title('4. Índice de Salud (HI) y Temperatura Operativa', fontweight='bold')
ax4.set_xlabel('Tiempo de Operación (Días)')
ax4.set_ylabel('Índice de Salud HI (%)', color='teal')
ax4_temp.set_ylabel('Temperatura (°C)', color='coral')
lines = p1 + p2
labels = [l.get_label() for l in lines]
ax4.legend(lines, labels, loc='lower left', fontsize=8)
ax4.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('/workspace/scratch/dashboard_gemelo_digital.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Dashboard generado correctamente.')


## 📋 5. Resumen de Estados Clave del Sistema
Resumen en tabla con la evolución del activo en días estratégicos:


In [ ]:
# Muestra del dataframe en días clave
dias_mostrar = [1, 25, 50, 75, 90, 100]
if dia_alerta and dia_alerta not in dias_mostrar:
    dias_mostrar.append(dia_alerta)
if dia_critico and dia_critico not in dias_mostrar:
    dias_mostrar.append(dia_critico)
dias_mostrar = sorted(list(set(dias_mostrar)))

resumen_dias = df_telemetria[df_telemetria['Dia'].isin(dias_mostrar)].copy()
resumen_dias['Vibracion_RMS_mms'] = resumen_dias['Vibracion_RMS_mms'].round(2)
resumen_dias['Temperatura_C'] = resumen_dias['Temperatura_C'].round(1)
resumen_dias['Indice_Salud_HI'] = resumen_dias['Indice_Salud_HI'].round(1)
resumen_dias = resumen_dias.rename(columns={
    'Dia': 'Día',
    'Vibracion_RMS_mms': 'Vibración RMS (mm/s)',
    'Temperatura_C': 'Temp (°C)',
    'Estado_ISO': 'Estado ISO 10816',
    'Indice_Salud_HI': 'Salud HI (%)',
    'RUL_Estimado_Dias': 'RUL Estimado (Días)'
})
print(resumen_dias[['Día', 'Vibración RMS (mm/s)', 'Temp (°C)', 'Estado ISO 10816', 'Salud HI (%)', 'RUL Estimado (Días)']].to_string(index=False))
